# HW12\n
\n
Компактный ноутбук для временных рядов: temporal split, baseline-модели и GRU.\n
\n
Ожидается файл `./data/S12-hw-dataset.csv` с колонками `date` и `target`.

In [ ]:
from pathlib import Path\n
import copy\n
import json\n
import math\n
import random\n
\n
import matplotlib.pyplot as plt\n
import numpy as np\n
import pandas as pd\n
import torch\n
import torch.nn as nn\n
from sklearn.linear_model import Ridge\n
from sklearn.metrics import mean_absolute_error, mean_squared_error\n
from sklearn.preprocessing import StandardScaler\n
from torch.utils.data import DataLoader, Dataset\n
\n
BASE_DIR = Path('.')\n
DATA_PATH = BASE_DIR / 'data' / 'S12-hw-dataset.csv'\n
ARTIFACTS_DIR = BASE_DIR / 'artifacts'\n
FIGURES_DIR = ARTIFACTS_DIR / 'figures'\n
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)\n
FIGURES_DIR.mkdir(parents=True, exist_ok=True)\n
\n
SEED = 42\n
random.seed(SEED)\n
np.random.seed(SEED)\n
torch.manual_seed(SEED)\n
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n
print('device =', DEVICE)\n
print('data_path =', DATA_PATH.resolve())

In [ ]:
if not DATA_PATH.exists():\n
    raise FileNotFoundError(f'Не найден датасет: {DATA_PATH}. Положи S12-hw-dataset.csv в папку data/.')\n
\n
df = pd.read_csv(DATA_PATH)\n
if 'date' not in df.columns or 'target' not in df.columns:\n
    raise ValueError('В датасете обязательно должны быть колонки date и target.')\n
\n
df['date'] = pd.to_datetime(df['date'])\n
df = df.sort_values('date').reset_index(drop=True)\n
df['target'] = pd.to_numeric(df['target'], errors='coerce')\n
df = df.dropna(subset=['target']).reset_index(drop=True)\n
\n
n_total = len(df)\n
train_end = int(n_total * 0.70)\n
val_end = int(n_total * 0.85)\n
train_df = df.iloc[:train_end].copy()\n
val_df = df.iloc[train_end:val_end].copy()\n
test_df = df.iloc[val_end:].copy()\n
\n
print('shape =', df.shape)\n
print('train/val/test =', len(train_df), len(val_df), len(test_df))\n
display(df.head())\n
\n
fig, ax = plt.subplots(figsize=(12, 4))\n
ax.plot(df['date'], df['target'], label='target', linewidth=1.5)\n
ax.axvspan(df.loc[0, 'date'], df.loc[train_end - 1, 'date'], alpha=0.12, color='tab:green', label='train')\n
ax.axvspan(df.loc[train_end, 'date'], df.loc[val_end - 1, 'date'], alpha=0.12, color='tab:orange', label='val')\n
ax.axvspan(df.loc[val_end, 'date'], df.loc[n_total - 1, 'date'], alpha=0.12, color='tab:red', label='test')\n
ax.set_title('Temporal split')\n
ax.legend()\n
ax.grid(alpha=0.2)\n
fig.tight_layout()\n
fig.savefig(FIGURES_DIR / 'series_split.png', dpi=150, bbox_inches='tight')\n
plt.show()

In [ ]:
def metrics(y_true, y_pred):\n
    y_true = np.asarray(y_true, dtype=float)\n
    y_pred = np.asarray(y_pred, dtype=float)\n
    mae = mean_absolute_error(y_true, y_pred)\n
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))\n
    denom = np.clip(np.abs(y_true), 1e-8, None)\n
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)\n
    return {'mae': float(mae), 'rmse': float(rmse), 'mape': float(mape)}\n
\n
def add_time_features(frame):\n
    out = frame.copy()\n
    out['lag_1'] = out['target'].shift(1)\n
    out['lag_7'] = out['target'].shift(7)\n
    out['lag_14'] = out['target'].shift(14)\n
    out['lag_28'] = out['target'].shift(28)\n
    out['roll_mean_7'] = out['target'].shift(1).rolling(7).mean()\n
    out['roll_mean_14'] = out['target'].shift(1).rolling(14).mean()\n
    out['dayofweek'] = out['date'].dt.dayofweek\n
    out['month'] = out['date'].dt.month\n
    return out\n
\n
feat_df = add_time_features(df).dropna().reset_index(drop=True)\n
feat_train = feat_df[feat_df['date'] <= train_df['date'].iloc[-1]].copy()\n
feat_val = feat_df[(feat_df['date'] >= val_df['date'].iloc[0]) & (feat_df['date'] <= val_df['date'].iloc[-1])].copy()\n
feat_test = feat_df[feat_df['date'] >= test_df['date'].iloc[0]].copy()\n
feature_cols = ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_mean_14', 'dayofweek', 'month']\n
\n
def naive_last(frame):\n
    return frame['lag_1'].to_numpy()\n
\n
def moving_average(frame):\n
    return frame['roll_mean_7'].to_numpy()\n
\n
ridge_scaler = StandardScaler()\n
X_train = ridge_scaler.fit_transform(feat_train[feature_cols])\n
y_train = feat_train['target'].to_numpy()\n
X_val = ridge_scaler.transform(feat_val[feature_cols])\n
X_test = ridge_scaler.transform(feat_test[feature_cols])\n
ridge = Ridge(alpha=1.0)\n
ridge.fit(X_train, y_train)\n
\n
baseline_rows = []\n
for experiment_id, title, val_pred, test_pred, notes in [\n
    ('B1', 'naive-last', naive_last(feat_val), naive_last(feat_test), 'predict t-1'),\n
    ('B2', 'moving-average', moving_average(feat_val), moving_average(feat_test), 'predict rolling mean(7)'),\n
    ('B3', 'ridge-lags', ridge.predict(X_val), ridge.predict(X_test), 'Ridge on lag and calendar features'),\n
]:\n
    val_metrics = metrics(feat_val['target'], val_pred)\n
    test_metrics = metrics(feat_test['target'], test_pred)\n
    baseline_rows.append({\n
        'experiment_id': experiment_id,\n
        'task': 'timeseries-forecasting',\n
        'dataset': 'S12-hw-dataset.csv',\n
        'seed': SEED,\n
        'split_summary': '70/15/15 temporal split',\n
        'window_size': 28,\n
        'horizon': 1,\n
        'model_summary': title,\n
        'features_summary': 'target lags + calendar' if experiment_id == 'B3' else 'target history only',\n
        'scaler': 'StandardScaler' if experiment_id == 'B3' else '',\n
        'optimizer': '',\n
        'lr': '',\n
        'epochs_trained': 0,\n
        'best_val_mae': val_metrics['mae'],\n
        'best_val_rmse': val_metrics['rmse'],\n
        'best_val_mape': val_metrics['mape'],\n
        'test_mae': test_metrics['mae'],\n
        'test_rmse': test_metrics['rmse'],\n
        'test_mape': test_metrics['mape'],\n
        'notes': notes,\n
    })\n
\n
baseline_df = pd.DataFrame(baseline_rows)\n
display(baseline_df)\n
\n
fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n
axes[0].bar(baseline_df['experiment_id'], baseline_df['best_val_mae'], color=['#5B8FF9', '#61DDAA', '#65789B'])\n
axes[0].set_title('Validation MAE')\n
axes[0].grid(axis='y', alpha=0.2)\n
axes[1].bar(baseline_df['experiment_id'], baseline_df['best_val_rmse'], color=['#5B8FF9', '#61DDAA', '#65789B'])\n
axes[1].set_title('Validation RMSE')\n
axes[1].grid(axis='y', alpha=0.2)\n
fig.tight_layout()\n
fig.savefig(FIGURES_DIR / 'baselines_compare.png', dpi=150, bbox_inches='tight')\n
plt.show()

In [ ]:
WINDOW_SIZE = 28\n
HORIZON = 1\n
BATCH_SIZE = 32\n
HIDDEN_SIZE = 64\n
NUM_LAYERS = 2\n
DROPOUT = 0.2\n
LR = 5e-4\n
MAX_EPOCHS = 30\n
\n
series = df['target'].to_numpy(dtype=np.float32)\n
train_mean = float(train_df['target'].mean())\n
train_std = float(train_df['target'].std() + 1e-8)\n
series_scaled = (series - train_mean) / train_std\n
\n
class WindowDataset(Dataset):\n
    def __init__(self, series_scaled, raw_series, target_indices, window_size):\n
        self.series_scaled = series_scaled\n
        self.raw_series = raw_series\n
        self.target_indices = target_indices\n
        self.window_size = window_size\n
\n
    def __len__(self):\n
        return len(self.target_indices)\n
\n
    def __getitem__(self, idx):\n
        target_idx = self.target_indices[idx]\n
        x = self.series_scaled[target_idx - self.window_size:target_idx]\n
        y_scaled = self.series_scaled[target_idx]\n
        y_raw = self.raw_series[target_idx]\n
        return (\n
            torch.tensor(x, dtype=torch.float32).unsqueeze(-1),\n
            torch.tensor(y_scaled, dtype=torch.float32),\n
            torch.tensor(y_raw, dtype=torch.float32),\n
        )\n
\n
all_indices = np.arange(WINDOW_SIZE, len(df))\n
train_indices = all_indices[all_indices < train_end]\n
val_indices = all_indices[(all_indices >= train_end) & (all_indices < val_end)]\n
test_indices = all_indices[all_indices >= val_end]\n
\n
train_ds = WindowDataset(series_scaled, series, train_indices, WINDOW_SIZE)\n
val_ds = WindowDataset(series_scaled, series, val_indices, WINDOW_SIZE)\n
test_ds = WindowDataset(series_scaled, series, test_indices, WINDOW_SIZE)\n
\n
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)\n
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)\n
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)\n
xb, yb, yraw = next(iter(train_loader))\n
print('x.shape =', tuple(xb.shape), 'y.shape =', tuple(yb.shape))\n
\n
class GRUForecast(nn.Module):\n
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):\n
        super().__init__()\n
        self.gru = nn.GRU(\n
            input_size=input_size,\n
            hidden_size=hidden_size,\n
            num_layers=num_layers,\n
            dropout=dropout if num_layers > 1 else 0.0,\n
            batch_first=True,\n
        )\n
        self.head = nn.Linear(hidden_size, 1)\n
\n
    def forward(self, x):\n
        out, _ = self.gru(x)\n
        return self.head(out[:, -1, :]).squeeze(-1)\n
\n
model = GRUForecast(hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT).to(DEVICE)\n
criterion = nn.MSELoss()\n
optimizer = torch.optim.Adam(model.parameters(), lr=LR)\n
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)\n
\n
def evaluate_model(model, loader):\n
    model.eval()\n
    losses = []\n
    preds = []\n
    targets = []\n
    with torch.no_grad():\n
        for xb, yb, yraw in loader:\n
            xb = xb.to(DEVICE)\n
            yb = yb.to(DEVICE)\n
            pred_scaled = model(xb)\n
            loss = criterion(pred_scaled, yb)\n
            pred_raw = pred_scaled.cpu().numpy() * train_std + train_mean\n
            losses.append(float(loss.item()))\n
            preds.extend(pred_raw.tolist())\n
            targets.extend(yraw.numpy().tolist())\n
    result = metrics(targets, preds)\n
    result['loss'] = float(np.mean(losses)) if losses else float('nan')\n
    result['preds'] = np.asarray(preds)\n
    result['targets'] = np.asarray(targets)\n
    return result\n
\n
history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': []}\n
best_state = None\n
best_val_loss = float('inf')\n
best_epoch = 0\n
\n
for epoch in range(1, MAX_EPOCHS + 1):\n
    model.train()\n
    batch_losses = []\n
    for xb, yb, _ in train_loader:\n
        xb = xb.to(DEVICE)\n
        yb = yb.to(DEVICE)\n
        optimizer.zero_grad()\n
        pred = model(xb)\n
        loss = criterion(pred, yb)\n
        loss.backward()\n
        optimizer.step()\n
        batch_losses.append(float(loss.item()))\n
\n
    train_loss = float(np.mean(batch_losses))\n
    val_result = evaluate_model(model, val_loader)\n
    scheduler.step(val_result['loss'])\n
\n
    history['train_loss'].append(train_loss)\n
    history['val_loss'].append(val_result['loss'])\n
    history['val_mae'].append(val_result['mae'])\n
    history['val_rmse'].append(val_result['rmse'])\n
\n
    if val_result['loss'] < best_val_loss:\n
        best_val_loss = val_result['loss']\n
        best_epoch = epoch\n
        best_state = copy.deepcopy(model.state_dict())\n
\n
    if epoch == 1 or epoch % 5 == 0 or epoch == MAX_EPOCHS:\n
        print('epoch={:02d} train_loss={:.4f} val_loss={:.4f} val_mae={:.4f}'.format(epoch, train_loss, val_result['loss'], val_result['mae']))\n
\n
model.load_state_dict(best_state)\n
torch.save(model.state_dict(), ARTIFACTS_DIR / 'best_gru.pt')\n
val_result = evaluate_model(model, val_loader)\n
test_result = evaluate_model(model, test_loader)\n
print('best_epoch =', best_epoch)\n
print('val metrics =', {k: round(v, 4) for k, v in val_result.items() if k in ['mae', 'rmse', 'mape']})\n
print('test metrics =', {k: round(v, 4) for k, v in test_result.items() if k in ['mae', 'rmse', 'mape']})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n
axes[0].plot(history['train_loss'], label='train_loss')\n
axes[0].plot(history['val_loss'], label='val_loss')\n
axes[0].set_title('GRU loss')\n
axes[0].legend()\n
axes[0].grid(alpha=0.2)\n
axes[1].plot(history['val_mae'], label='val_mae')\n
axes[1].plot(history['val_rmse'], label='val_rmse')\n
axes[1].set_title('GRU validation metrics')\n
axes[1].legend()\n
axes[1].grid(alpha=0.2)\n
fig.tight_layout()\n
fig.savefig(FIGURES_DIR / 'gru_learning_curves.png', dpi=150, bbox_inches='tight')\n
plt.show()\n
\n
test_dates = df.loc[test_indices, 'date'].to_numpy()\n
fig, ax = plt.subplots(figsize=(12, 4))\n
ax.plot(test_dates, test_result['targets'], label='actual', linewidth=1.8)\n
ax.plot(test_dates, test_result['preds'], label='gru_pred', linewidth=1.5)\n
ax.set_title('Best forecast on test')\n
ax.legend()\n
ax.grid(alpha=0.2)\n
fig.tight_layout()\n
fig.savefig(FIGURES_DIR / 'best_forecast_test.png', dpi=150, bbox_inches='tight')\n
plt.show()\n
\n
gru_row = {\n
    'experiment_id': 'R1',\n
    'task': 'timeseries-forecasting',\n
    'dataset': 'S12-hw-dataset.csv',\n
    'seed': SEED,\n
    'split_summary': '70/15/15 temporal split',\n
    'window_size': WINDOW_SIZE,\n
    'horizon': HORIZON,\n
    'model_summary': 'GRU(hidden=64, layers=2, dropout=0.2)',\n
    'features_summary': 'univariate target windows',\n
    'scaler': 'train mean/std',\n
    'optimizer': 'Adam',\n
    'lr': LR,\n
    'epochs_trained': best_epoch,\n
    'best_val_mae': val_result['mae'],\n
    'best_val_rmse': val_result['rmse'],\n
    'best_val_mape': val_result['mape'],\n
    'test_mae': test_result['mae'],\n
    'test_rmse': test_result['rmse'],\n
    'test_mape': test_result['mape'],\n
    'notes': 'best checkpoint by validation loss',\n
}\n
\n
runs_df = pd.concat([baseline_df, pd.DataFrame([gru_row])], ignore_index=True)\n
runs_df.to_csv(ARTIFACTS_DIR / 'runs.csv', index=False)\n
\n
config = {\n
    'experiment_id': 'R1',\n
    'model': 'gru-forecast',\n
    'seed': SEED,\n
    'window_size': WINDOW_SIZE,\n
    'architecture': {\n
        'input_size': 1,\n
        'hidden_size': HIDDEN_SIZE,\n
        'num_layers': NUM_LAYERS,\n
        'dropout': DROPOUT,\n
    },\n
    'training': {\n
        'batch_size': BATCH_SIZE,\n
        'learning_rate': LR,\n
        'epochs': MAX_EPOCHS,\n
        'best_epoch': best_epoch,\n
        'optimizer': 'Adam',\n
        'loss': 'MSELoss',\n
        'scheduler': 'ReduceLROnPlateau',\n
    },\n
    'features': {\n
        'target_column': 'target',\n
        'date_column': 'date',\n
        'horizon': HORIZON,\n
    },\n
    'scaler': {\n
        'name': 'train_mean_std',\n
        'train_mean': train_mean,\n
        'train_std': train_std,\n
    },\n
}\n
with open(ARTIFACTS_DIR / 'best_gru_config.json', 'w', encoding='utf-8') as f:\n
    json.dump(config, f, ensure_ascii=False, indent=2)\n
\n
final_eval = {\n
    'best_epoch': best_epoch,\n
    'val': {k: float(val_result[k]) for k in ['mae', 'rmse', 'mape']},\n
    'test': {k: float(test_result[k]) for k in ['mae', 'rmse', 'mape']},\n
}\n
with open(ARTIFACTS_DIR / 'final_test_evaluation.json', 'w', encoding='utf-8') as f:\n
    json.dump(final_eval, f, ensure_ascii=False, indent=2)\n
\n
display(runs_df)